### Code Hist.

 - CODE  
    &ensp; : Crawling - 특일 정보 조회 (KASI)

  - DATE  
    &ensp; 2023-11-29 Created  
    &emsp;&emsp;&emsp;&emsp;&emsp;&emsp; 1)   
    &emsp;&emsp;&emsp;&emsp;&emsp;&emsp; 2)   
    &emsp;&emsp;&emsp;&emsp;&emsp;&emsp; 3)   
    
 - DESC  
    &ensp; : 전처리 - 한국지역난방공사 열판매량/열공급량   
    &emsp; 1) 결측치가 없어서, 그대로 사용  
    &emsp;&ensp;&ensp; 
    &emsp;&ensp;&ensp; (Crawl Code 없음)   

 - DATA  
    &emsp; <"Input">  
    1) None (Input Dataset)  
    &emsp;- Period :   
    &emsp;- Interval : 

    &emsp; <"Output">  
    1) Hourly (관측소/년도별 출력)  
    &nbsp;df_data_cal.to_csv(data_dir + 'KASI_DATE_D_Final.csv', index = False, encoding='utf-8-sig')  
    &emsp;- Columns : ['YEAR', 'MONTH', 'DAY'  
    &emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;, 'dateKind', 'code_day_of_the_week', 'day_of_the_week'  
    &emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;, 'rest_YN', 'name_of_holiday', 'dist_from_holiday']
    &emsp;- Period :   
    &emsp;- Interval :  
    
    2) Daily (관측소/년도별 출력)  
    &nbsp;df_data_cal_24.to_csv(data_dir + 'KASI_DATE_H_Final.csv', index = False, encoding='utf-8-sig')  
    &emsp;- Columns : ['locdate', 'YEAR', 'MONTH', 'DAY'  
    &emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;, 'dateKind', 'code_day_of_the_week', 'day_of_the_week'  
    &emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;, 'rest_YN', 'name_of_holiday', 'dist_from_holiday'  
    &emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&emsp;, 'HOUR', 'MINUTE']
    &emsp;- Period :   
    &emsp;- Interval :  
    
    

 - Related Link  
    &ensp; : 

# 01. Code

## 01-01. Init

### 01-01-01. Init_Module Import

In [ ]:
#region Basic_Import
## Basic
import os
os.path.dirname(os.path.abspath('__file__'))
import sys
sys.path.append(os.path.dirname(os.path.abspath(os.path.dirname('__file__'))))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pandas import DataFrame, Series

import math
import random

## Datetime
import time
import datetime as dt
from datetime import datetime, date, timedelta

import glob
from glob import glob
import requests
import json

## 시각화
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
plt.rcParams['figure.figsize'] = [10, 8]

from scipy import stats

# K-Means 알고리즘
from sklearn.cluster import KMeans, MiniBatchKMeans
from sklearn.model_selection import train_test_split

# CLustering 알고리즘의 성능 평가 측도
from sklearn.metrics import homogeneity_score, completeness_score, v_measure_score, adjusted_rand_score, silhouette_score, rand_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.metrics.cluster import contingency_matrix

## 정규화
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn import metrics

import urllib
from urllib.request import urlopen
from urllib.parse import urlencode, unquote, quote_plus

from selenium import webdriver
from selenium.webdriver.chrome.service import Service

from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup

## Init.
pd.options.display.float_format = '{:.10f}'.format
#endregion Basic_Import

In [ ]:
## Import_DL
str_tar = "tf"
## For Torch
if str_tar == "torch":
    import torch
    import torch.nn as nn
    from torch.nn.utils import weight_norm
    print("Torch Imported")
## For TF
elif str_tar == "tf":
    import tensorflow as tf
    import tensorflow_addons as tfa
    print("Tensorflow Imported")
else:
    print("Error : Cannot be used except for Keywords")
    print(" : torch / tf")

In [ ]:
## Import_Local
from core import data_datetime as com_date
from core import provider_kasi as com_Holi
from core import data_analysis as com_Analysis
from core import data_preprocessing as com_Prep
from core import data_visualization as com_Visual
from core import provider_kma as com_ASOS
from core import provider_kdhc as com_KDHC
from core import provider_kier as com_KIER

### 01-01-02. Config (Directory, Params)

In [ ]:
## Init_config
SEED = 42

np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ['TF_DETERMINISTIC_OPS'] = "1"
random.seed(SEED)

In [ ]:
## Define Todate str
str_now_ymd = pd.datetime.now().date()
str_now_y = pd.datetime.now().year
str_now_m = pd.datetime.now().month
str_now_d = pd.datetime.now().day
str_now_hr = pd.datetime.now().hour
str_now_min = pd.datetime.now().minute

print(pd.datetime.now())
print(str(str_now_y) + " / " + str(str_now_m)  + " / " + str(str_now_d))
print(str(str_now_hr) + " : " + str(str_now_min))

In [ ]:
## Dict_Domain
dict_domain = {0:"ELEC", 1:"HEAT", 2:"WATER", 3:"HOT_HEAT", 4:"HOT_FLOW", 99:"GAS"} ## GAS는 사용하지 않음.
int_domain = 0
str_domain = str(dict_domain[int_domain])

dict_col_accu = {0 : "ACTUAL_ACCU_EFF" ## ELEC
                 , 1 : "ACCU_HEAT" ## HEAT
                 , 2 : "ACCU_FLOW" ## WATER
                 , 3 : "ACCU_HEAT" ## HOT 열량
                 , 4 : "ACCU_FLOW" ## HOT 유량
                 , 99 : "ACCU_FLOW" ## GAS
                 }
str_col_accu = str(str_domain + "_" + str(dict_col_accu[int_domain]))

dict_col_inst = {0 : "INST_EFF" ## ELEC_ACCU/INST_EFF
                , 1 : "INST_HEAT" ## HEAT_ACCU/INST_HEAT
                , 2 : "INST_FLOW" ## WATER_ACCU/INST_FLOW
                , 3 : "INST_HEAT" ## HOT_ACCU/INST_HEAT
                , 4 : "INST_FLOW" ## HOT_ACCU/INST_FLOW
                , 99 : "INST_FLOW" ## GAS_ACCU/INST_FLOW
                } 
str_col_inst = str(str_domain + "_" + str(dict_col_inst[int_domain]))

## Directory Root
str_dirData = "../data/data_Energy_KIER/"
str_dir_raw = '../data/data_Energy_KIER/KIER_0_Raw/'
str_dirName_bld = '../data/data_Energy_KIER/KIER_1_BLD/'
str_dirName_f = '../data/data_Energy_KIER/KIER_2_F_' + str_domain + '/'
str_dirName_h = '../data/data_Energy_KIER/KIER_3_H_' + str_domain + '/'

## File
str_fileRaw = str('KIER_RAW_' + str_domain + '_2023-11-12.csv')
str_fileRaw_hList = str('KIER_hList_' + str_domain + '.csv')

print(str(os.listdir(str_dirData)) + "\n")
print(os.listdir(str_dirName_h))

## 01-02. Data Load (df_raw)

### 01-02-01. KDHC Heat Usage (Intergrated)

In [ ]:
## 10분 단위
str_file = 'KIER_' + str_domain + '_INST_03-01_10min.csv'
## 1시간 단위
# str_file = 'KIER_' + str_domain + '_INST_03-02_1H.csv'
## 1일 단위
# str_file = 'KIER_' + str_domain + '_INST_03-03_1D.csv' 
## 1주 단위
# str_file = 'KIER_' + str_domain + '_INST_03-04_1W.csv' 
## 1개월 단위
# str_file = 'KIER_' + str_domain + '_INST_03-05_1M.csv' 
## [미구현] 분기 단위
# str_file = 'KIER_' + str_domain + '_ACCU_MAXACCU_InstBaseUpdated.csv' 
## [미사용] 기간 총합
# str_file = 'KIER_' + str_domain + '_ACCU_MAXACCU_InstBaseUpdated.csv' 

str_interval = str_file[len('KIER_' + str_domain + '_INST_'):-len('_InstBaseUpdated.csv')]

df_kier_raw = pd.read_csv(str_dirName_h + str_file
                          , index_col = 0)
df_kier_raw['METER_DATE'] = pd.to_datetime(df_kier_raw['METER_DATE'])
df_kier_raw

In [ ]:
## 호실별 순시 사용량 컬럼만 가져오기
list_col = df_kier_raw.columns[1:]
# df_kier_h = df_kier_raw[list_col]
# df_kier_h = df_kier_raw.drop(columns = ['index', 'YEAR', 'MONTH', 'DAY'
#                                       , 'HOUR', 'MINUTE'
#                                       , 'MEAN_OF_INST', 'SUM_OF_INST'])
# df_kier_h = df_kier_h.set_index('METER_DATE')

df_kier_h = df_kier_raw.set_index('METER_DATE')

## [미사용] 기간 총합
# df_kier_h = df_kier_raw.set_index('H_INDEX')
df_kier_h

In [ ]:
df_kier_summary_total = df_kier_h.transpose()
df_kier_summary_total = df_kier_summary_total.reset_index()

## 세대 번호의 컬럼명이 'index'로 지정되어 오류 발생
df_kier_summary_total['h_index'] = df_kier_summary_total['index']
df_kier_summary_total = df_kier_summary_total.drop(columns = ['index'])

df_kier_summary_total

In [ ]:
X = df_kier_summary_total.drop(columns = 'h_index')
y = df_kier_summary_total['h_index']

In [ ]:
X

http://bigdata.dongguk.ac.kr/lectures/datascience/_book/%EA%B5%B0%EC%A7%91%EB%B6%84%EC%84%9D.html

In [ ]:
# 변수 표준화
scaler = StandardScaler() # 변수 표준화 클래스
scaler.fit(X)  # 표준화를 위해 변수별 파라미터(평균, 표준편차) 계산
# scaler.mean_, scaler.scale_
X_std = scaler.transform(X)  # 훈련자료 표준화 변환
X_std

### Clustering (군집화) : K - Means

군집의 수 결정 방법: elbow method - 군집의 개수와 군집내 변동의 합을 그래프로 나타내고, 변동량의 변화가 작아지는 지점의 군집의 수를 적정 군집의 수로 결정함

In [ ]:
Objectives = []
K = range(2,15)
for k in K:
    km = KMeans(n_clusters=k)
    km = km.fit(X_std)
    Objectives.append(km.inertia_)

plt.plot(K, Objectives, 'bx-')
plt.xlabel('k')
plt.ylabel('Objectives')
plt.xlabel('number of clusters')
plt.show() 
print(Objectives)
Obj_deriv = []
for i in range(1, len(Objectives)):
    Obj_deriv.append(abs(float(Objectives[i]) - float(Objectives[i - 1])))
print(Obj_deriv)

In [ ]:
def get_calinski_harabasz_index(X, cluster):
    cluster_label = np.unique(cluster) ## 클러스터 라벨
    K = len(cluster_label) ## 총 클러스터 개수
    n = X.shape[0] ## 총 데이터 개수
    c = np.mean(X, axis=0) ## 전체 데이터 중심 벡터
    num_sum = 0 ## calinski_harabasz_index 분자
    denom_sum = 0 ## calinski_harabasz_index 분모
    for cl in cluster_label:
        sub_X = X[np.where(cluster==cl)[0], :]
        c_k = np.mean(sub_X, axis=0)
        n_k = sub_X.shape[0]
        num_sum += n_k*np.sum(np.square(c_k-c))
        denom_sum += np.sum(np.square(sub_X-c_k))
        
    ## calinski_harabasz_index
    calinski_harabasz_index = (num_sum/(K-1))/(denom_sum/(n-K))
    return calinski_harabasz_index

In [ ]:
# kmeans = KMeans(n_clusters=k, random_state=0).fit(X_std)
cluster = km.predict(X_std)
print(get_calinski_harabasz_index(X_std, cluster))

In [ ]:
n_cluster_list = [1, 2, 3, 4, 5, 6, 7, 8, 9, 10]

chi_list = []
K = range(2,15)
for n_cluster in n_cluster_list:
    km_chi = KMeans(n_clusters = n_cluster
                    , init="k-means++"
                    , max_iter=300
                    , n_init=1)
    km_chi.fit(X_std) 
    cluster = km_chi.predict(X_std)
    chi_list.append(get_calinski_harabasz_index(X_std, cluster))

fig = plt.figure(figsize=(8,8))
fig.set_facecolor('white')
ax = fig.add_subplot()
ax.plot(n_cluster_list, chi_list)
ax.set_xticks(n_cluster_list)
plt.show()

In [ ]:
K = 2

# ## 군집의 수 변화를 파악하기 위해 10번 반복
# list_log_clusters = [] ## 군집의 수 총 기록
# for int_phase in range(0, 10):
#     list_cnt_clusters = [] ## Init_군집별 개체 수
#     km = KMeans(n_clusters = K
#                 , init="k-means++"
#                 , max_iter=300
#                 , n_init=1)

#     ## k-means alogorithm 적합
#     km.fit(X_std) 

#     ## 결과: 레코드별 군집 라벨
#     print(km.labels_)
#     for i in range(0, K):
#         int_size_cluster = len(np.where(km.labels_ == i)[0])
#         list_cnt_clusters.append(int_size_cluster)
#         print(int_size_cluster)

#     print()
#     # 결과: 군집별 컬럼별 중심평균
#     # print(km.cluster_centers_)
#     # 목적함수의 값
#     # print(km.inertia_)

#     list_log_clusters.append(list_cnt_clusters)
#     int_phase = int_phase + 1

# print(list_log_clusters)

In [ ]:
K = 3

km = KMeans(n_clusters = K
            , init="k-means++"
            , max_iter=300
            , n_init=1)

## k-means alogorithm 적합
km.fit(X_std) 

## 결과: 레코드별 군집 라벨
print(km.labels_)
for i in range(0, K):
    int_size_cluster = len(np.where(km.labels_ == i)[0])
    print(int_size_cluster)

print()
# 결과: 군집별 컬럼별 중심평균
# print(km.cluster_centers_)
# 목적함수의 값
# print(km.inertia_)

In [ ]:
# df_kier_summary_total['target'] = np.transpose(np.where(km.labels_ == i)[0])
df_kier_summary_total['target_'+str_domain] = 0
for i in range(0, len(df_kier_summary_total)):
    df_kier_summary_total['target_'+str_domain].iloc[i] = km.labels_[i]
# df_kier_summary_total[['h_index', 'target_' + str_domain]]

str_file_labeled = str_dirName_h + 'KIER_' + str(str_domain) + '_Labeled_' + str_interval + '.csv'
df_kier_summary_total = df_kier_summary_total[['h_index', 'target_'+str_domain]]
df_kier_summary_total.to_csv(str_file_labeled)
df_kier_summary_total

In [ ]:
# Building별 Pattern scatter plot
# df_kier_summary_total['target'] = np.transpose(np.where(km.labels_ == i)[0])
dict_bld_c = {"561" : 0, "562" : 1, "563" : 2}
df_kier_summary_total['target_'+str_domain] = 0
for i in range(0, len(df_kier_summary_total)):
    df_kier_summary_total['target_'+str_domain].iloc[i] = dict_bld_c[str(df_kier_summary_total['h_index'].iloc[i])[len(str_col_inst) + 1:len(str_col_inst) + 4]]
# df_kier_summary_total[['h_index', 'target_' + str_domain]]

In [ ]:
km.labels_ = np.array(df_kier_summary_total['target_'+str_domain])
km.labels_

In [ ]:
str_file_label_by_bld = str_dirName_h + 'KIER_' + str(str_domain) + '_Label_by_Bld_' + str_interval + '.csv'
df_kier_sum_bld = df_kier_summary_total[['h_index', 'target_'+str_domain]]
df_kier_sum_bld.to_csv(str_file_label_by_bld)

cluster_bld = np.array(df_kier_sum_bld['target_'+str_domain].to_list())
color=['blue', 'green', 'orange', 'cyan', 'red', 'black', 'yellow', 'peru', 'purple', 'slategray']
for k in range(K):
    data = X_std[cluster == k]
    # plt.scatter(data[:, 0], data[:, 1], data[:, 2], c=color[k], alpha=0.8, label='cluster %d' % k)
    plt.scatter(data[:, 0], data[:, 1], c=color[k], alpha=0.8, label='bld %d' % k)
    plt.scatter(km.cluster_centers_[k, 0], km.cluster_centers_[k, 1], c='red', marker="x")
    plt.title('Clustering by buildings')
plt.legend(fontsize=12, loc='upper right') # legend position
plt.xlabel('X')
plt.ylabel('Y')
plt.show()

In [ ]:
## 모형 평가
labels = km.labels_

# print('contingency_matrix\n' , contingency_matrix(y, km.labels_))
print('Silhouette Coefficient: %.3f' % silhouette_score(X_std, labels, sample_size=1000))
print('Calinski and Harabasz score: %.3f' % calinski_harabasz_score(X_std, labels))
print("Davues Bouldin Score: %0.3f" % davies_bouldin_score(X_std, labels))

print("Homogeneity: %0.3f" % homogeneity_score(y, labels))
print("Completeness: %0.3f" % completeness_score(y, labels))
print("V-measure: %0.3f" % v_measure_score(y, labels))
print("Rand-Index: %0.3f" % rand_score(y, labels))
print("Adjusted Rand-Index: %.3f" % adjusted_rand_score(y, labels))